# Feature Engineering for Logistic Regression

## Overview
This notebook documents the feature engineering process applied to the particle dataset in preparation for downstream logistic regression modeling.

The objective is to construct features that:
- Improve linear separability between classes
- Capture meaningful elemental relationships
- Align with statistical assumptions of logistic regression

All transformations are informed by prior exploratory data analysis (EDA), including UMAP results and domain knowledge in forensic particle classification.

## Feature Engineering Design Considerations

Feature construction was guided by both statistical and domain-specific principles:

### 1. Linearity in Log-Odds
Logistic regression assumes a linear relationship between predictors and the log-odds of the outcome. Transformations are applied to better approximate this relationship.

### 2. Skew Reduction
Elemental measurements exhibit strong right-skew. Log transformations are used to stabilize distributions.

### 3. Relative Relationships
EDA and UMAP analysis indicate that relative composition (rather than absolute values) drives class separation. Ratio and percentage-based features are prioritized.

### 4. Confounder Adjustment
Environmental elements (e.g., Zn, Cu, Ti) introduce overlap between classes. Features are constructed to explicitly compare GSR-related elements against these confounders.

### 5. Interpretability
Feature complexity is constrained to maintain interpretability, which is critical for downstream analysis and potential forensic applications.

### 5. Simplicity & Interpretability
Feature complexity is intentionally constrained to ensure compatibility with logistic regression and maintain interpretability.

## Data Loading

The processed particle dataset is loaded from the project data directory.

In [9]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../../../data/processed/particle_labeled.parquet")

print(df.shape)
df.head()

(2801667, 95)


,stub_id,particle_id,relevance_class,ac,ag,al,ar,as,at,au,...,v,w,xe,y,yb,zn,zr,merged_relevance_class,final_class,label
0,22,1454,PbSb,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PbSb,PbSb,GSR
1,22,1274,PbSbBa,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PbSbBa,PbBaSb,GSR
2,22,275,PbSbBa,0.0,0.0,0.751013,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PbSbBa,PbBaSb,GSR
3,22,714,PbSbBa,0.0,0.0,0.824510,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PbSbBa,PbBaSb,GSR
4,22,2887,PbSb,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PbSb,PbSb,GSR


## Feature Engineering Approach

The following transformations are applied:

- Log transformations of elemental features
- Log-transformed elemental ratios (Pb, Ba, Sb)
- GSR-to-confounder comparison features
- Composition-based (% mass) features
- Domain-based indicator features

These features are designed to improve model stability and predictive performance while maintaining interpretability.

In [10]:
def engineer_features_logistic(df):
    df = df.copy()
    eps = 1e-6  # Prevent division by zero

    # -----------------------------
    # 1. Log transformations
    # -----------------------------
    for col in ["pb", "ba", "sb", "zn", "cu", "ti"]:
        df[f"log_{col}"] = np.log1p(df[col])

    # -----------------------------
    # 2. Elemental ratios (log scale)
    # -----------------------------
    df["log_pb_ba_ratio"] = np.log1p(df["pb"] / (df["ba"] + eps))
    df["log_pb_sb_ratio"] = np.log1p(df["pb"] / (df["sb"] + eps))
    df["log_ba_sb_ratio"] = np.log1p(df["ba"] / (df["sb"] + eps))

    # -----------------------------
    # 3. GSR vs confounders
    # -----------------------------
    gsr_sum = df["pb"] + df["ba"] + df["sb"]
    conf_sum = df["zn"] + df["cu"] + df["ti"]

    df["log_gsr_over_confounders"] = np.log1p(gsr_sum / (conf_sum + eps))

    # -----------------------------
    # 4. Composition features
    # -----------------------------
    total_mass = gsr_sum + conf_sum + eps

    df["pb_pct"] = df["pb"] / total_mass
    df["ba_pct"] = df["ba"] / total_mass
    df["sb_pct"] = df["sb"] / total_mass

    # -----------------------------
    # 5. Domain-based indicator
    # -----------------------------
    df["gsr_count"] = (
        (df["pb"] > 0).astype(int) +
        (df["ba"] > 0).astype(int) +
        (df["sb"] > 0).astype(int)
    )

    return df

## Apply Feature Engineering

The feature engineering function is applied to the dataset to generate the transformed feature set.

In [11]:
df_fe = engineer_features_logistic(df)

print(df_fe.shape)
df_fe.head()

(2801667, 109)


,stub_id,particle_id,relevance_class,ac,ag,al,ar,as,at,au,...,log_cu,log_ti,log_pb_ba_ratio,log_pb_sb_ratio,log_ba_sb_ratio,log_gsr_over_confounders,pb_pct,ba_pct,sb_pct,gsr_count
0,22,1454,PbSb,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.0,16.363694,0.220675,0.000000,17.983067,0.198023,0.000000,0.801977,2
1,22,1274,PbSbBa,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.784369,2.703078,2.540981,17.532948,0.523170,0.439261,0.037569,3
2,22,275,PbSbBa,0.0,0.0,0.751013,0.0,0.0,0.0,0.0,...,0.000000,0.0,1.283676,1.241908,0.664449,17.502325,0.558879,0.214139,0.226982,3
3,22,714,PbSbBa,0.0,0.0,0.824510,0.0,0.0,0.0,0.0,...,2.189444,0.0,0.977714,1.207655,0.881458,1.777103,0.409433,0.246888,0.174551,3
4,22,2887,PbSb,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.0,15.960464,0.277069,0.000000,17.379290,0.241998,0.000000,0.758002,2


## Feature Selection for Logistic Regression

A subset of engineered features is selected based on:

- Relevance to EDA findings
- Reduction of multicollinearity
- Interpretability
- Expected contribution to linear decision boundaries

These features will be used in downstream modeling.

In [12]:
features = [
    "log_pb", "log_ba", "log_sb",
    "log_pb_ba_ratio",
    "log_pb_sb_ratio",
    "log_ba_sb_ratio",
    "log_gsr_over_confounders",
    "pb_pct", "ba_pct", "sb_pct",
    "gsr_count"
]

target = "label"

X = df_fe[features]
y = df_fe[target]

print("Feature matrix shape:", X.shape)

Feature matrix shape: (2801667, 11)


## Save Engineered Dataset

The engineered dataset is saved for use in downstream modeling notebooks.

In [13]:
df_fe.to_parquet("../../../data/processed/engineered_features_logistic.parquet", index=False)

## Summary

This feature engineering pipeline transforms the raw particle dataset into a form suitable for logistic regression by:

- Reducing skew through log transformations
- Capturing key elemental relationships via ratios
- Representing relative composition through percentage features
- Incorporating domain-informed indicators

These transformations are designed to support stable, interpretable, and effective logistic regression modeling in subsequent analysis.